In [24]:
#imports
import pandas as pd
pd.set_option('display.max_columns', None)

In [25]:
crash_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Crashes_20260407.csv', delimiter=',')
vehicle_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Vehicles_20260503.csv', delimiter=',')
person_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Person_20260503.csv', delimiter=',')

/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_15220/1986786415.py:1: DtypeWarning: Columns (0: ZIP CODE) have mixed types. Specify dtype option on import or set low_memory=False.
  crash_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Crashes_20260407.csv', delimiter=',')
/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_15220/1986786415.py:2: DtypeWarning: Columns (0: VEHICLE_MODEL, 1: VEHICLE_OCCUPANTS) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Vehicles_20260503.csv', delimiter=',')
/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_15220/1986786415.py:3: DtypeWarning: Columns (0: PERSON_AGE) have mixed types. Specify dtype option on import or set low_memory=False.
  person_df = pd.read_csv('datasets/Motor_Vehicle_Collisions_-_Person_20260503.csv', delimiter=',')


In [43]:
#The size of the dataset in MB
print(f"Size of the crash dataset: {crash_df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
print(f"Size of the vehicle dataset: {vehicle_df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")
print(f"Size of the person dataset: {person_df.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")

Size of the crash dataset: 875.21 MB
Size of the vehicle dataset: 1605.91 MB
Size of the person dataset: 1883.19 MB


### Clean crash data

In [44]:
crash_clean_df = crash_df.copy()

In [45]:
crash_clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2253192 entries, 0 to 2253191
Data columns (total 29 columns):
 #   Column                         Dtype  
---  ------                         -----  
 0   CRASH DATE                     str    
 1   CRASH TIME                     str    
 2   BOROUGH                        str    
 3   ZIP CODE                       object 
 4   LATITUDE                       float64
 5   LONGITUDE                      float64
 6   LOCATION                       str    
 7   ON STREET NAME                 str    
 8   CROSS STREET NAME              str    
 9   OFF STREET NAME                str    
 10  NUMBER OF PERSONS INJURED      float64
 11  NUMBER OF PERSONS KILLED       float64
 12  NUMBER OF PEDESTRIANS INJURED  int64  
 13  NUMBER OF PEDESTRIANS KILLED   int64  
 14  NUMBER OF CYCLIST INJURED      int64  
 15  NUMBER OF CYCLIST KILLED       int64  
 16  NUMBER OF MOTORIST INJURED     int64  
 17  NUMBER OF MOTORIST KILLED      int64  
 18  CONTRIBUTING 

In [46]:
# range of dates in the dataset
print(crash_clean_df["CRASH DATE"].min())
print(crash_clean_df["CRASH DATE"].max())

01/01/2013
12/31/2025


In [47]:
# how many nan values
crash_clean_df.isna().sum()

CRASH DATE                             0
CRASH TIME                             0
BOROUGH                           687462
ZIP CODE                          687745
LATITUDE                          240676
LONGITUDE                         240676
LOCATION                          240676
ON STREET NAME                    494034
CROSS STREET NAME                 862706
OFF STREET NAME                  1851532
NUMBER OF PERSONS INJURED             18
NUMBER OF PERSONS KILLED              31
NUMBER OF PEDESTRIANS INJURED          0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF CYCLIST INJURED              0
NUMBER OF CYCLIST KILLED               0
NUMBER OF MOTORIST INJURED             0
NUMBER OF MOTORIST KILLED              0
CONTRIBUTING FACTOR VEHICLE 1       8152
CONTRIBUTING FACTOR VEHICLE 2     365477
CONTRIBUTING FACTOR VEHICLE 3    2090099
CONTRIBUTING FACTOR VEHICLE 4    2215899
CONTRIBUTING FACTOR VEHICLE 5    2242979
COLLISION_ID                           0
VEHICLE TYPE COD

In [48]:
print(crash_clean_df['ZIP CODE'].nunique())
print(crash_clean_df['BOROUGH'].nunique())

438
5


In [49]:
# are the missing values in zip code and in borough are the same rows? what about lat lon columns?
missing_zip = crash_clean_df['ZIP CODE'].isna()
missing_borough = crash_clean_df['BOROUGH'].isna()
missing_lat = crash_clean_df['LATITUDE'].isna()
missing_lon = crash_clean_df['LONGITUDE'].isna()

print(f"Missing ZIP CODE:    {missing_zip.sum()}")
print(f"Missing BOROUGH:     {missing_borough.sum()}")
print(f"Missing both zip and borough:            {(missing_zip & missing_borough).sum()}")
print(f"Missing zip but NOT borough:             {(missing_zip & ~missing_borough).sum()}")
print(f"Missing borough but NOT zip:             {(missing_borough & ~missing_zip).sum()}")
pct = (missing_zip & missing_borough).sum() / missing_zip.sum() * 100
print(f"\nOf rows with missing ZIP, {pct:.1f}% also have missing BOROUGH")

print(f"\nMissing zip AND missing coords:          {(missing_zip & missing_lat).sum()}")
print(f"Missing zip but has valid coordinates:   {(missing_zip & ~missing_lat & ~missing_lon).sum()}")
print(f"Missing borough but has valid coords:    {(missing_borough & ~missing_lat & ~missing_lon).sum()}")

Missing ZIP CODE:    687745
Missing BOROUGH:     687462
Missing both zip and borough:            687462
Missing zip but NOT borough:             283
Missing borough but NOT zip:             0

Of rows with missing ZIP, 100.0% also have missing BOROUGH

Missing zip AND missing coords:          202847
Missing zip but has valid coordinates:   484898
Missing borough but has valid coords:    484641


#### Dropping nan values

As we will be working with geographical features of the data, we need to have consistency across columns such as borough, zip code, and coordinates - rows with NaNs in coordinates will be dropped, nans in zip code and borough could be inferred from the coordinates, as most of the rows with missing lat and lon have non-nan values.

For the features describing the collisions, we are not dropping NaN values,as it is expected that some refering to several vehicles will be empty, if it was a one-vehicle accident.

We will also drop the rows with nan values in number of persons injured and killed, as these features should be fully usable.

As the number of missing values in the columns containing street info would significantly further reduce the size of the dataset, while it is not sure if these features will be used, we decide to drop these features.

Nan values in contributing factor columns and vehicle type codes will be left unchanged, as it is expected for many values to be missing (not all crashes involve cars or a specific number of cars)

#### Filling missing boroughs

Most rows with a missing ZIP CODE also have a missing BOROUGH. However, many of these rows have valid latitude/longitude coordinates. We infer the borough from the coordinates using a KNN classifier trained on rows that already have both coordinates and borough.

In [50]:
from sklearn.neighbors import KNeighborsClassifier

has_coords = (
    crash_clean_df['LATITUDE'].notna() &
    crash_clean_df['LONGITUDE'].notna()
)

def knn_fill(df, feature_cols, target_col, has_coords_mask, n_neighbors=5):
    train = df[has_coords_mask & df[target_col].notna()]
    fill_idx = df.index[has_coords_mask & df[target_col].isna()]
    print(f"{target_col} — training rows: {len(train)}, rows to fill: {len(fill_idx)}")
    if len(fill_idx) == 0:
        print(f"  Nothing to fill.")
        return
    knn = KNeighborsClassifier(n_neighbors=n_neighbors)
    # cast target to str so sklearn treats it as discrete classes,
    # not continuous — needed when the column dtype is float (e.g. ZIP CODE)
    knn.fit(train[feature_cols], train[target_col].astype(str))
    df.loc[fill_idx, target_col] = knn.predict(df.loc[fill_idx, feature_cols])
    print(f"  Missing after fill: {df[target_col].isna().sum()}")

knn_fill(crash_clean_df, ['LATITUDE', 'LONGITUDE'], 'BOROUGH', has_coords)
knn_fill(crash_clean_df, ['LATITUDE', 'LONGITUDE'], 'ZIP CODE', has_coords)

BOROUGH — training rows: 1527875, rows to fill: 484641
  Missing after fill: 202821
ZIP CODE — training rows: 1527618, rows to fill: 484898
  Missing after fill: 202847


In [51]:
# drop rows missing injury/kill counts — these features must be present
# rows with missing coordinates, zip code, or borough are kept
# (borough and zip were inferred from coordinates where possible above)
crash_clean_df = crash_clean_df.dropna(subset=['NUMBER OF PERSONS INJURED', 'NUMBER OF PERSONS KILLED'])
# crash_clean_df = crash_clean_df.dropna(subset=['ZIP CODE', 'LATITUDE', 'LONGITUDE'])
crash_clean_df = crash_clean_df.drop(['ON STREET NAME', 'CROSS STREET NAME', 'OFF STREET NAME'], axis=1)
crash_clean_df.isna().sum()

CRASH DATE                             0
CRASH TIME                             0
BOROUGH                           202818
ZIP CODE                          202844
LATITUDE                          240673
LONGITUDE                         240673
LOCATION                          240673
NUMBER OF PERSONS INJURED              0
NUMBER OF PERSONS KILLED               0
NUMBER OF PEDESTRIANS INJURED          0
NUMBER OF PEDESTRIANS KILLED           0
NUMBER OF CYCLIST INJURED              0
NUMBER OF CYCLIST KILLED               0
NUMBER OF MOTORIST INJURED             0
NUMBER OF MOTORIST KILLED              0
CONTRIBUTING FACTOR VEHICLE 1       8145
CONTRIBUTING FACTOR VEHICLE 2     365447
CONTRIBUTING FACTOR VEHICLE 3    2090062
CONTRIBUTING FACTOR VEHICLE 4    2215862
CONTRIBUTING FACTOR VEHICLE 5    2242942
COLLISION_ID                           0
VEHICLE TYPE CODE 1                16830
VEHICLE TYPE CODE 2               457797
VEHICLE TYPE CODE 3              2096495
VEHICLE TYPE COD

In [52]:
# remove duplicates
crash_clean_df = crash_clean_df.drop_duplicates()

In [53]:
print(crash_clean_df['ZIP CODE'].unique())

[nan 11230.0 11208.0 11233.0 '11211.0' 10475.0 11207.0 10017.0 '11385.0'
 11413.0 '11214' 11434.0 11217.0 11226.0 10463.0 '11357.0' 10001.0
 '11364.0' 11372.0 '10035.0' 10301.0 11215.0 11211.0 10455.0 11385.0
 '10128.0' 11418.0 11225.0 '11379.0' '10314' '10462.0' '10029.0' 11220.0
 11411.0 10452.0 10466.0 '10033.0' '10032.0' 10453.0 10019.0 '11217.0'
 11221.0 '10023.0' '11233.0' 11203.0 '11209.0' 11419.0 11101.0 11106.0
 '11691.0' 11223.0 '10453.0' 11422.0 '11210.0' 11213.0 10128.0 11218.0
 11692.0 11420.0 '11249.0' '10309.0' 11205.0 11212.0 10022.0 10011.0
 10314.0 '10463' '11225.0' '11223.0' 10461.0 '11203.0' '11101.0' '10025.0'
 11004.0 10025.0 11373.0 10018.0 11234.0 10462.0 10472.0 11206.0 11236.0
 11210.0 '10454.0' 11238.0 11209.0 10065.0 11249.0 11432.0 '10001.0'
 10032.0 '11436.0' '10470.0' 11104.0 10002.0 '11238.0' '10466.0' '10022.0'
 10456.0 '10468.0' '10303' 10468.0 11201.0 11219.0 '11229.0' '11216.0'
 '10452.0' 11222.0 11235.0 10012.0 '11378.0' '10302' 10305.0 '11434.0'
 1

In [54]:
# zip code cleanup: normalize floats like 11230.0 → '11230', strip whitespace,
# then keep only valid 5-digit NYC zip codes starting with 10 or 11, and convert to int
crash_clean_df['ZIP CODE'] = (
    crash_clean_df['ZIP CODE']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
    .str.strip()
)
crash_clean_df = crash_clean_df[crash_clean_df['ZIP CODE'].str.match(r'^(10|11)\d{3}$', na=False)]
crash_clean_df['ZIP CODE'] = crash_clean_df['ZIP CODE'].astype(int)

In [55]:
crash_clean_df['ZIP CODE'].nunique()

234

In [56]:
# for convenience we will convert the date and time columns to datetime format
crash_clean_df['CRASH DATE'] = pd.to_datetime(crash_clean_df['CRASH DATE']).dt.date
crash_clean_df['CRASH TIME'] = pd.to_datetime(crash_clean_df['CRASH TIME']).dt.time
crash_clean_df

/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_15220/2091243873.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crash_clean_df['CRASH TIME'] = pd.to_datetime(crash_clean_df['CRASH TIME']).dt.time


,CRASH DATE,CRASH TIME,BOROUGH,ZIP CODE,LATITUDE,LONGITUDE,LOCATION,NUMBER OF PERSONS INJURED,NUMBER OF PERSONS KILLED,NUMBER OF PEDESTRIANS INJURED,NUMBER OF PEDESTRIANS KILLED,NUMBER OF CYCLIST INJURED,NUMBER OF CYCLIST KILLED,NUMBER OF MOTORIST INJURED,NUMBER OF MOTORIST KILLED,CONTRIBUTING FACTOR VEHICLE 1,CONTRIBUTING FACTOR VEHICLE 2,CONTRIBUTING FACTOR VEHICLE 3,CONTRIBUTING FACTOR VEHICLE 4,CONTRIBUTING FACTOR VEHICLE 5,COLLISION_ID,VEHICLE TYPE CODE 1,VEHICLE TYPE CODE 2,VEHICLE TYPE CODE 3,VEHICLE TYPE CODE 4,VEHICLE TYPE CODE 5
2,2023-11-01,01:29:00,BROOKLYN,11230,40.621790,-73.970024,"(40.62179, -73.970024)",1.0,0.0,0,0,0,0,1,0,Unspecified,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN
9,2021-09-11,09:35:00,BROOKLYN,11208,40.667202,-73.866500,"(40.667202, -73.8665)",0.0,0.0,0,0,0,0,0,0,Unspecified,NaN,NaN,NaN,NaN,4456314,Sedan,NaN,NaN,NaN,NaN
10,2021-12-14,08:13:00,BROOKLYN,11233,40.683304,-73.917274,"(40.683304, -73.917274)",0.0,0.0,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,4486609,NaN,NaN,NaN,NaN,NaN
12,2021-12-14,17:05:00,BROOKLYN,11211,40.709183,-73.956825,"(40.709183, -73.956825)",0.0,0.0,0,0,0,0,0,0,Passing Too Closely,Unspecified,NaN,NaN,NaN,4486555,Sedan,Tractor Truck Diesel,NaN,NaN,NaN
13,2021-12-14,08:17:00,BRONX,10475,40.868160,-73.831480,"(40.86816, -73.83148)",2.0,0.0,0,0,0,0,2,0,Unspecified,Unspecified,NaN,NaN,NaN,4486660,Sedan,Sedan,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2253186,2021-04-20,20:20:00,STATEN ISLAND,10303,40.622635,-74.160810,"(40.622635, -74.16081)",0.0,0.0,0,0,0,0,0,0,Driver Inattention/Distraction,Unspecified,NaN,NaN,NaN,4408988,Station Wagon/Sport Utility Vehicle,NaN,NaN,NaN,NaN
2253188,2021-04-20,15:07:00,BROOKLYN,11238,40.683480,-73.966896,"(40.68348, -73.966896)",1.0,0.0,0,0,0,0,1,0,Unspecified,Unspecified,NaN,NaN,NaN,4408962,Convertible,Moped,NaN,NaN,NaN
2253189,2021-04-20,17:00:00,BRONX,10458,40.855600,-73.885150,"(40.8556, -73.88515)",0.0,0.0,0,0,0,0,0,0,Passing or Lane Usage Improper,Unspecified,NaN,NaN,NaN,4409125,Sedan,NaN,NaN,NaN,NaN
2253190,2021-04-20,16:45:00,BROOKLYN,11204,40.625828,-73.990950,"(40.625828, -73.99095)",0.0,0.0,0,0,0,0,0,0,Unspecified,NaN,NaN,NaN,NaN,4408955,Sedan,NaN,NaN,NaN,NaN


### Clean vehicle data

In [57]:
vehicle_filtered_df = vehicle_df[vehicle_df['COLLISION_ID'].isin(crash_clean_df['COLLISION_ID'])]
vehicle_filtered_df.info()

<class 'pandas.DataFrame'>
Index: 4110814 entries, 0 to 4530509
Data columns (total 25 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   UNIQUE_ID                    int64  
 1   COLLISION_ID                 int64  
 2   CRASH_DATE                   str    
 3   CRASH_TIME                   str    
 4   VEHICLE_ID                   str    
 5   STATE_REGISTRATION           str    
 6   VEHICLE_TYPE                 str    
 7   VEHICLE_MAKE                 str    
 8   VEHICLE_MODEL                str    
 9   VEHICLE_YEAR                 float64
 10  TRAVEL_DIRECTION             str    
 11  VEHICLE_OCCUPANTS            object 
 12  DRIVER_SEX                   str    
 13  DRIVER_LICENSE_STATUS        str    
 14  DRIVER_LICENSE_JURISDICTION  str    
 15  PRE_CRASH                    str    
 16  POINT_OF_IMPACT              str    
 17  VEHICLE_DAMAGE               str    
 18  VEHICLE_DAMAGE_1             str    
 19  VEHICLE_DAMAGE_2

In [58]:
vehicle_filtered_df.isna().sum()

UNIQUE_ID                            0
COLLISION_ID                         0
CRASH_DATE                           0
CRASH_TIME                           0
VEHICLE_ID                           0
STATE_REGISTRATION              350571
VEHICLE_TYPE                    266598
VEHICLE_MAKE                   1690732
VEHICLE_MODEL                  4065278
VEHICLE_YEAR                   1715393
TRAVEL_DIRECTION               1443276
VEHICLE_OCCUPANTS              1572708
DRIVER_SEX                     2071726
DRIVER_LICENSE_STATUS          2171706
DRIVER_LICENSE_JURISDICTION    2168714
PRE_CRASH                       824527
POINT_OF_IMPACT                1477003
VEHICLE_DAMAGE                 1504493
VEHICLE_DAMAGE_1               2416561
VEHICLE_DAMAGE_2               2816377
VEHICLE_DAMAGE_3               3098587
PUBLIC_PROPERTY_DAMAGE         1286860
PUBLIC_PROPERTY_DAMAGE_TYPE    4081451
CONTRIBUTING_FACTOR_1           161572
CONTRIBUTING_FACTOR_2          1464423
dtype: int64

Lot of missing data, but the dataset is big - more than 4 million rows, so the columns will not be dropped, as they might be needed differently based on the analysis.

In [59]:
# for convenience we will convert the date and time columns to datetime format
vehicle_filtered_df['CRASH_DATE'] = pd.to_datetime(vehicle_filtered_df['CRASH_DATE']).dt.date
vehicle_filtered_df['CRASH_TIME'] = pd.to_datetime(vehicle_filtered_df['CRASH_TIME']).dt.time
vehicle_filtered_df

/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_15220/236786060.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  vehicle_filtered_df['CRASH_TIME'] = pd.to_datetime(vehicle_filtered_df['CRASH_TIME']).dt.time


,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,VEHICLE_ID,STATE_REGISTRATION,VEHICLE_TYPE,VEHICLE_MAKE,VEHICLE_MODEL,VEHICLE_YEAR,TRAVEL_DIRECTION,VEHICLE_OCCUPANTS,DRIVER_SEX,DRIVER_LICENSE_STATUS,DRIVER_LICENSE_JURISDICTION,PRE_CRASH,POINT_OF_IMPACT,VEHICLE_DAMAGE,VEHICLE_DAMAGE_1,VEHICLE_DAMAGE_2,VEHICLE_DAMAGE_3,PUBLIC_PROPERTY_DAMAGE,PUBLIC_PROPERTY_DAMAGE_TYPE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2
0,10385780,100201,2012-09-07,09:03:00,1,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
1,19140702,4213082,2019-09-23,08:15:00,0553ab4d-9500-4cba-8d98-f4d7f89d5856,NY,Station Wagon/Sport Utility Vehicle,TOYT -CAR/SUV,NaN,2002.0,North,1.0,M,Licensed,NY,Going Straight Ahead,Left Front Bumper,Left Front Quarter Panel,NaN,NaN,NaN,N,NaN,Driver Inattention/Distraction,Unspecified
2,14887647,3307608,2015-10-02,17:18:00,2,NY,TAXI,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Going Straight Ahead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Driver Inattention/Distraction,NaN
5,17044639,3434155,2016-05-02,17:35:00,219456,NY,4 dr sedan,MERZ -CAR/SUV,NaN,2015.0,East,2.0,M,Licensed,FL,Merging,Right Front Bumper,Right Front Bumper,Right Front Quarter Panel,NaN,NaN,N,NaN,Driver Inattention/Distraction,Unsafe Lane Changing
6,19138701,4229067,2019-10-24,13:15:00,c53b43d9-419a-4ab1-9361-3f2979078d89,NY,Bus,FRHT-TRUCK/BUS,NaN,2006.0,East,13.0,M,Licensed,NY,Parked,Left Front Quarter Panel,Left Front Quarter Panel,NaN,NaN,NaN,N,NaN,Unspecified,Unspecified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4530108,21170701,4885023,2026-03-11,21:10:00,94f0b7bc-9cd8-40ee-9eea-0a8f4477c373,DC,Sedan,TESL -CAR/SUV,NaN,2022.0,East,1.0,F,Licensed,WA,Parked,Left Rear Bumper,Left Rear Bumper,Left Rear Bumper,Left Rear Bumper,Left Rear Bumper,N,NaN,Unspecified,Unspecified
4530113,21170785,4883751,2026-03-06,20:40:00,6d84f993-c950-48cd-ac55-eba2fa822ace,NY,Sedan,INFI -CAR/SUV,NaN,2015.0,South,3.0,M,Licensed,NY,Going Straight Ahead,Center Front End,Center Front End,Left Front Bumper,Right Front Bumper,NaN,N,NaN,Unsafe Speed,Unsafe Speed
4530319,21170699,4885023,2026-03-11,21:10:00,04f21d1a-df8c-4cf9-b5cb-d18c60be000a,NY,Sedan,TOYT -CAR/SUV,NaN,2022.0,East,1.0,M,Licensed,NY,Going Straight Ahead,Left Front Bumper,Left Front Bumper,Left Front Bumper,Left Front Bumper,Center Front End,N,NaN,Unspecified,Driver Inattention/Distraction
4530380,21170691,4879817,2026-02-18,10:15:00,b88ddf69-0d6c-4365-a67a-611c97e92c48,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN,NaN,NaN


In [60]:
vehicle_filtered_df['VEHICLE_TYPE'].nunique()

2984

In [61]:
# vehicle clean up

def clean_occupants(val):
    if pd.isna(val):
        return pd.NA
    
    # Convert to string, strip whitespace
    val = str(val).strip()
    
    # Empty string
    if val == "":
        return pd.NA
    
    # Remove commas from numbers like '999,999' or '1,355'
    val = val.replace(",", "")
    
    # Try converting to float first (handles '1.0' style strings)
    try:
        val = float(val)
    except ValueError:
        return pd.NA
    
    # Cap unrealistic values
    if val > 1000 or val < 0:
        return pd.NA
    
    return int(val)

vehicle_filtered_df["VEHICLE_OCCUPANTS"] = vehicle_filtered_df["VEHICLE_OCCUPANTS"].apply(clean_occupants).astype("Int64")

In [62]:
vehicle_filtered_df['VEHICLE_OCCUPANTS'].unique()

<IntegerArray>
[<NA>,    1,    2,   13,    3,    0,    5,    4,    7,   33,
 ...
  737,  111,  743,  101,  135,   75,  169,   91,   80,  349]
Length: 110, dtype: Int64

In [63]:
# remove duplicates
vehicle_filtered_df = vehicle_filtered_df.drop_duplicates()

### Clean person data

In [64]:
person_filtered_df = person_df[person_df['COLLISION_ID'].isin(crash_clean_df['COLLISION_ID'])]
person_filtered_df.info()

<class 'pandas.DataFrame'>
Index: 5540409 entries, 0 to 5948789
Data columns (total 21 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   UNIQUE_ID              int64  
 1   COLLISION_ID           int64  
 2   CRASH_DATE             str    
 3   CRASH_TIME             str    
 4   PERSON_ID              str    
 5   PERSON_TYPE            str    
 6   PERSON_INJURY          str    
 7   VEHICLE_ID             float64
 8   PERSON_AGE             object 
 9   EJECTION               str    
 10  EMOTIONAL_STATUS       str    
 11  BODILY_INJURY          str    
 12  POSITION_IN_VEHICLE    str    
 13  SAFETY_EQUIPMENT       str    
 14  PED_LOCATION           str    
 15  PED_ACTION             str    
 16  COMPLAINT              str    
 17  PED_ROLE               str    
 18  CONTRIBUTING_FACTOR_1  str    
 19  CONTRIBUTING_FACTOR_2  str    
 20  PERSON_SEX             str    
dtypes: float64(1), int64(2), object(1), str(17)
memory usage: 1.6+ GB


In [65]:
person_filtered_df.isna().sum()

UNIQUE_ID                      0
COLLISION_ID                   0
CRASH_DATE                     0
CRASH_TIME                     0
PERSON_ID                     19
PERSON_TYPE                    0
PERSON_INJURY                  0
VEHICLE_ID                235963
PERSON_AGE                621864
EJECTION                 2698562
EMOTIONAL_STATUS         2600011
BODILY_INJURY            2599969
POSITION_IN_VEHICLE      2698122
SAFETY_EQUIPMENT         2899481
PED_LOCATION             5436676
PED_ACTION               5436774
COMPLAINT                2599962
PED_ROLE                  165419
CONTRIBUTING_FACTOR_1    5437911
CONTRIBUTING_FACTOR_2    5438042
PERSON_SEX                601471
dtype: int64

Same as with the vehicle dataset, lot of missing values in many columns, but we will handle them accordingly to analysis needs.

In [66]:
# for convenience we will convert the date and time columns to datetime format
person_filtered_df['CRASH_DATE'] = pd.to_datetime(person_filtered_df['CRASH_DATE']).dt.date
person_filtered_df['CRASH_TIME'] = pd.to_datetime(person_filtered_df['CRASH_TIME']).dt.time
person_filtered_df

/var/folders/tb/jb06w8hn5r72136sx1ztbxnr0000gn/T/ipykernel_15220/1529943711.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  person_filtered_df['CRASH_TIME'] = pd.to_datetime(person_filtered_df['CRASH_TIME']).dt.time


,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,EMOTIONAL_STATUS,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,PED_LOCATION,PED_ACTION,COMPLAINT,PED_ROLE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2,PERSON_SEX
0,10249006,4229554,2019-10-26,09:43:00,31aa2bc0-f545-444f-8cdb-f1cb5cf00b89,Occupant,Unspecified,19141108.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,U
1,10255054,4230587,2019-10-25,15:15:00,4629e500-a73e-48dc-b8fb-53124d124b80,Occupant,Unspecified,19144075.0,33,Not Ejected,Does Not Apply,Does Not Apply,"Front passenger, if two or more persons, inclu...",Lap Belt & Harness,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,F
2,10253177,4230550,2019-10-26,17:55:00,ae48c136-1383-45db-83f4-2a5eecfb7cff,Occupant,Unspecified,19143133.0,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,M
3,6650180,3565527,2016-11-21,13:05:00,2782525,Occupant,Unspecified,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Notified Person,NaN,NaN,NaN
5,10253606,4230743,2019-10-24,19:15:00,84bcb3a7-d201-4c61-9e30-fe29268c1074,Occupant,Injured,19143343.0,27,Not Ejected,Conscious,Back,Driver,Lap Belt & Harness,NaN,NaN,Complaint of Pain or Nausea,Driver,NaN,NaN,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5948681,13881046,4883751,2026-03-06,20:40:00,d9282230-e3ec-4705-b1c3-a232316aff61,Occupant,Unspecified,21170785.0,16.0,Not Ejected,Does Not Apply,Does Not Apply,"Middle rear seat, or passenger lying across a ...",Lap Belt & Harness,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,M
5948694,13880881,4885023,2026-03-11,21:10:00,c3a55acc-c79e-48b5-aeb9-d0db390eb011,Occupant,Unspecified,21170701.0,35.0,Not Ejected,Does Not Apply,Does Not Apply,Driver,Unknown,NaN,NaN,Does Not Apply,Driver,NaN,NaN,F
5948715,13880880,4885023,2026-03-11,21:10:00,bcd89acb-66b6-4b6e-8a8a-9b4b2da39ee2,Occupant,Unspecified,21170699.0,59.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,M
5948784,13881048,4883751,2026-03-06,20:40:00,bf54f013-a260-435b-875b-c81278ed99ba,Occupant,Unspecified,21170785.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,U


In [67]:
# remove duplicates
person_filtered_df = person_filtered_df.drop_duplicates()

In [68]:
person_filtered_df['PERSON_AGE'].unique()

array([nan, '33', '55', ..., '1,108', '-215', '-745'],
      shape=(1031,), dtype=object)

In [69]:
# person age clean up

def clean_person_age(val):
    if pd.isna(val):
        return pd.NA
    
    # Convert to string, strip whitespace
    val = str(val).strip()
    
    # Empty string
    if val == "":
        return pd.NA
    
    # Remove commas from numbers like '999,999' or '1,355'
    val = val.replace(",", "")
    
    # Try converting to float first (handles '1.0' style strings)
    try:
        val = float(val)
    except ValueError:
        return pd.NA
    
    # Cap unrealistic values
    if val > 120 or val < 0:
        return pd.NA
    
    return int(val)

person_filtered_df["PERSON_AGE"] = person_filtered_df["PERSON_AGE"].apply(clean_person_age).astype("Int64")

### Merging


In [70]:
# compare vehicle_id in vehicle and person datasets
print(vehicle_filtered_df['VEHICLE_ID'].dtype)
print(person_filtered_df['VEHICLE_ID'].dtype)
vehicle_filtered_df.head()

str
float64


,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,VEHICLE_ID,STATE_REGISTRATION,VEHICLE_TYPE,VEHICLE_MAKE,VEHICLE_MODEL,VEHICLE_YEAR,TRAVEL_DIRECTION,VEHICLE_OCCUPANTS,DRIVER_SEX,DRIVER_LICENSE_STATUS,DRIVER_LICENSE_JURISDICTION,PRE_CRASH,POINT_OF_IMPACT,VEHICLE_DAMAGE,VEHICLE_DAMAGE_1,VEHICLE_DAMAGE_2,VEHICLE_DAMAGE_3,PUBLIC_PROPERTY_DAMAGE,PUBLIC_PROPERTY_DAMAGE_TYPE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2
0,10385780,100201,2012-09-07,09:03:00,1,NY,PASSENGER VEHICLE,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Unspecified,NaN
1,19140702,4213082,2019-09-23,08:15:00,0553ab4d-9500-4cba-8d98-f4d7f89d5856,NY,Station Wagon/Sport Utility Vehicle,TOYT -CAR/SUV,NaN,2002.0,North,1,M,Licensed,NY,Going Straight Ahead,Left Front Bumper,Left Front Quarter Panel,NaN,NaN,NaN,N,NaN,Driver Inattention/Distraction,Unspecified
2,14887647,3307608,2015-10-02,17:18:00,2,NY,TAXI,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,Going Straight Ahead,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Driver Inattention/Distraction,NaN
5,17044639,3434155,2016-05-02,17:35:00,219456,NY,4 dr sedan,MERZ -CAR/SUV,NaN,2015.0,East,2,M,Licensed,FL,Merging,Right Front Bumper,Right Front Bumper,Right Front Quarter Panel,NaN,NaN,N,NaN,Driver Inattention/Distraction,Unsafe Lane Changing
6,19138701,4229067,2019-10-24,13:15:00,c53b43d9-419a-4ab1-9361-3f2979078d89,NY,Bus,FRHT-TRUCK/BUS,NaN,2006.0,East,13,M,Licensed,NY,Parked,Left Front Quarter Panel,Left Front Quarter Panel,NaN,NaN,NaN,N,NaN,Unspecified,Unspecified


In [71]:
person_filtered_df.head()

,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,EMOTIONAL_STATUS,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,PED_LOCATION,PED_ACTION,COMPLAINT,PED_ROLE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2,PERSON_SEX
0,10249006,4229554,2019-10-26,09:43:00,31aa2bc0-f545-444f-8cdb-f1cb5cf00b89,Occupant,Unspecified,19141108.0,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,U
1,10255054,4230587,2019-10-25,15:15:00,4629e500-a73e-48dc-b8fb-53124d124b80,Occupant,Unspecified,19144075.0,33,Not Ejected,Does Not Apply,Does Not Apply,"Front passenger, if two or more persons, inclu...",Lap Belt & Harness,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,F
2,10253177,4230550,2019-10-26,17:55:00,ae48c136-1383-45db-83f4-2a5eecfb7cff,Occupant,Unspecified,19143133.0,55,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,M
3,6650180,3565527,2016-11-21,13:05:00,2782525,Occupant,Unspecified,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Notified Person,NaN,NaN,NaN
5,10253606,4230743,2019-10-24,19:15:00,84bcb3a7-d201-4c61-9e30-fe29268c1074,Occupant,Injured,19143343.0,27,Not Ejected,Conscious,Back,Driver,Lap Belt & Harness,NaN,NaN,Complaint of Pain or Nausea,Driver,NaN,NaN,M


In [72]:
vehicle_filtered_df[vehicle_filtered_df['COLLISION_ID'] == 4229067]

,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,VEHICLE_ID,STATE_REGISTRATION,VEHICLE_TYPE,VEHICLE_MAKE,VEHICLE_MODEL,VEHICLE_YEAR,TRAVEL_DIRECTION,VEHICLE_OCCUPANTS,DRIVER_SEX,DRIVER_LICENSE_STATUS,DRIVER_LICENSE_JURISDICTION,PRE_CRASH,POINT_OF_IMPACT,VEHICLE_DAMAGE,VEHICLE_DAMAGE_1,VEHICLE_DAMAGE_2,VEHICLE_DAMAGE_3,PUBLIC_PROPERTY_DAMAGE,PUBLIC_PROPERTY_DAMAGE_TYPE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2
6,19138701,4229067,2019-10-24,13:15:00,c53b43d9-419a-4ab1-9361-3f2979078d89,NY,Bus,FRHT-TRUCK/BUS,NaN,2006.0,East,13,M,Licensed,NY,Parked,Left Front Quarter Panel,Left Front Quarter Panel,NaN,NaN,NaN,N,NaN,Unspecified,Unspecified
1326,19138700,4229067,2019-10-24,13:15:00,8630cd86-1c74-48b0-96e5-e3b306d9fd1d,NY,Bus,IVEC-TRUCK/BUS,NaN,2006.0,East,33,F,Licensed,NY,Going Straight Ahead,Right Front Quarter Panel,Right Front Quarter Panel,NaN,NaN,NaN,Unspecified,NaN,Driver Inattention/Distraction,Unspecified


In [73]:
person_filtered_df[person_filtered_df['COLLISION_ID'] == 4229067].head(5)

,UNIQUE_ID,COLLISION_ID,CRASH_DATE,CRASH_TIME,PERSON_ID,PERSON_TYPE,PERSON_INJURY,VEHICLE_ID,PERSON_AGE,EJECTION,EMOTIONAL_STATUS,BODILY_INJURY,POSITION_IN_VEHICLE,SAFETY_EQUIPMENT,PED_LOCATION,PED_ACTION,COMPLAINT,PED_ROLE,CONTRIBUTING_FACTOR_1,CONTRIBUTING_FACTOR_2,PERSON_SEX
111758,10244109,4229067,2019-10-24,13:15:00,d278534c-d3c3-4055-86af-6499b04a58d6,Occupant,Unspecified,19138700.0,8,Not Ejected,Does Not Apply,Does Not Apply,"Any person in the rear of a station wagon, pic...",Unknown,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,F
122916,10244119,4229067,2019-10-24,13:15:00,17d3c04f-73fc-4206-983c-c7ba5dbcde8b,Occupant,Unspecified,19138700.0,10,Not Ejected,Does Not Apply,Does Not Apply,"Any person in the rear of a station wagon, pic...",Unknown,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,F
127965,10244108,4229067,2019-10-24,13:15:00,073152fe-f2f5-4311-9d09-cc49982e009f,Occupant,Unspecified,19138700.0,8,Not Ejected,Does Not Apply,Does Not Apply,"Any person in the rear of a station wagon, pic...",Unknown,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,M
174901,10244107,4229067,2019-10-24,13:15:00,05c2d822-a48c-4198-901e-7dcea7d67bbf,Occupant,Unspecified,19138700.0,7,Not Ejected,Does Not Apply,Does Not Apply,"Any person in the rear of a station wagon, pic...",Unknown,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,M
220392,10244105,4229067,2019-10-24,13:15:00,9e87b621-33ce-4581-958b-0e3f42721a52,Occupant,Unspecified,19138700.0,8,Not Ejected,Does Not Apply,Does Not Apply,"Any person in the rear of a station wagon, pic...",Unknown,NaN,NaN,Does Not Apply,Passenger,NaN,NaN,M


Vehicle_id in person dataset is the unique_id in vehicle_id, so the datasets can be merged using this link.

In [74]:
print(person_filtered_df['VEHICLE_ID'].dtype)
print(vehicle_filtered_df['UNIQUE_ID'].dtype)

float64
int64


In [75]:
# convert vehicle_id in person dataset to int if the value is numeric, otherwise it will be converted to NaN
person_filtered_df['VEHICLE_ID'] = person_filtered_df['VEHICLE_ID'].astype('Int64')
print(person_filtered_df['VEHICLE_ID'].dtype)
print(vehicle_filtered_df['UNIQUE_ID'].dtype)

Int64
int64


In [76]:
# add to every column name in vehicle dataset the prefix 'v_' to avoid confusion when merging with person dataset
crash_clean_df = crash_clean_df.add_prefix('c_')
vehicle_filtered_df = vehicle_filtered_df.add_prefix('v_')
person_filtered_df = person_filtered_df.add_prefix('p_')

In [77]:
# save each cleaned dataset separately with COLLISION_ID as index
# use these files as the source of truth in analysis notebooks
crash_clean_df.set_index("c_COLLISION_ID").to_parquet(
    "datasets/crashes_clean.parquet", engine="pyarrow"
)
vehicle_filtered_df.set_index("v_COLLISION_ID").to_parquet(
    "datasets/vehicles_clean.parquet", engine="pyarrow"
)
person_filtered_df.set_index("p_COLLISION_ID").to_parquet(
    "datasets/persons_clean.parquet", engine="pyarrow"
)
print(f"crashes_clean:  {len(crash_clean_df):,} rows")
print(f"vehicles_clean: {len(vehicle_filtered_df):,} rows")
print(f"persons_clean:  {len(person_filtered_df):,} rows")

crashes_clean:  2,050,108 rows
vehicles_clean: 4,110,814 rows
persons_clean:  5,540,409 rows


In [78]:
crash_vehicle_df = crash_clean_df.merge(vehicle_filtered_df, left_on="c_COLLISION_ID", right_on='v_COLLISION_ID',  how="left")

In [79]:
full_df = crash_vehicle_df.merge(
    person_filtered_df,
    left_on=["c_COLLISION_ID", "v_UNIQUE_ID"],
    right_on=["p_COLLISION_ID", "p_VEHICLE_ID"],
    how="left"
)
full_df.columns

Index(['c_CRASH DATE', 'c_CRASH TIME', 'c_BOROUGH', 'c_ZIP CODE', 'c_LATITUDE',
       'c_LONGITUDE', 'c_LOCATION', 'c_NUMBER OF PERSONS INJURED',
       'c_NUMBER OF PERSONS KILLED', 'c_NUMBER OF PEDESTRIANS INJURED',
       'c_NUMBER OF PEDESTRIANS KILLED', 'c_NUMBER OF CYCLIST INJURED',
       'c_NUMBER OF CYCLIST KILLED', 'c_NUMBER OF MOTORIST INJURED',
       'c_NUMBER OF MOTORIST KILLED', 'c_CONTRIBUTING FACTOR VEHICLE 1',
       'c_CONTRIBUTING FACTOR VEHICLE 2', 'c_CONTRIBUTING FACTOR VEHICLE 3',
       'c_CONTRIBUTING FACTOR VEHICLE 4', 'c_CONTRIBUTING FACTOR VEHICLE 5',
       'c_COLLISION_ID', 'c_VEHICLE TYPE CODE 1', 'c_VEHICLE TYPE CODE 2',
       'c_VEHICLE TYPE CODE 3', 'c_VEHICLE TYPE CODE 4',
       'c_VEHICLE TYPE CODE 5', 'v_UNIQUE_ID', 'v_COLLISION_ID',
       'v_CRASH_DATE', 'v_CRASH_TIME', 'v_VEHICLE_ID', 'v_STATE_REGISTRATION',
       'v_VEHICLE_TYPE', 'v_VEHICLE_MAKE', 'v_VEHICLE_MODEL', 'v_VEHICLE_YEAR',
       'v_TRAVEL_DIRECTION', 'v_VEHICLE_OCCUPANTS', 'v_DR

In [80]:
# Should match original crash count if all crashes are preserved
assert full_df["c_COLLISION_ID"].nunique() == crash_clean_df["c_COLLISION_ID"].nunique()

# Spot-check pedestrians: null p_VEHICLE_ID but valid collision_id
pedestrians = full_df[full_df["p_VEHICLE_ID"].isna()]
print(f"Pedestrian-involved rows: {len(pedestrians)}")

# Check for unexpected row explosion (a sign of a bad join)
print(f"Crashes: {len(crash_clean_df)}, Final rows: {len(full_df)}")

Pedestrian-involved rows: 1532396
Crashes: 2050108, Final rows: 6836842


### Renaming columns and dropping duplicating data

In [81]:
full_df[['c_CRASH DATE', 'c_CRASH TIME', 'v_CRASH_DATE', 'v_CRASH_TIME', 'p_CRASH_DATE', 'p_CRASH_TIME']].isna().sum()

c_CRASH DATE          0
c_CRASH TIME          0
v_CRASH_DATE       1821
v_CRASH_TIME       1821
p_CRASH_DATE    1530666
p_CRASH_TIME    1530666
dtype: int64

In [82]:
full_df[['v_UNIQUE_ID', 'p_UNIQUE_ID']].isna().sum()

v_UNIQUE_ID       1821
p_UNIQUE_ID    1530666
dtype: int64

The missing values for vehicles, are the crashes that have no matching vehicle record. The missing values for person are the crash, vehicle pair that had no match to any (crash, vehicle) pair.

In [83]:
# flag columns to indicate if a crash has a vehicle or person record
full_df["HAS_VEHICLE"] = full_df["v_UNIQUE_ID"].notna().astype(int)
full_df["HAS_PERSON"] = full_df["p_UNIQUE_ID"].notna().astype(int)

In [84]:
# let's drop all spare columns
cols_to_drop = [
    "v_COLLISION_ID", "p_COLLISION_ID",
    "p_VEHICLE_ID", "v_VEHICLE_ID",     
    "v_CRASH_DATE", "v_CRASH_TIME",
    "p_CRASH_DATE", "p_CRASH_TIME",
]
full_df = full_df.drop(columns=cols_to_drop)

In [85]:
full_df.head()

,c_CRASH DATE,c_CRASH TIME,c_BOROUGH,c_ZIP CODE,c_LATITUDE,c_LONGITUDE,c_LOCATION,c_NUMBER OF PERSONS INJURED,c_NUMBER OF PERSONS KILLED,c_NUMBER OF PEDESTRIANS INJURED,c_NUMBER OF PEDESTRIANS KILLED,c_NUMBER OF CYCLIST INJURED,c_NUMBER OF CYCLIST KILLED,c_NUMBER OF MOTORIST INJURED,c_NUMBER OF MOTORIST KILLED,c_CONTRIBUTING FACTOR VEHICLE 1,c_CONTRIBUTING FACTOR VEHICLE 2,c_CONTRIBUTING FACTOR VEHICLE 3,c_CONTRIBUTING FACTOR VEHICLE 4,c_CONTRIBUTING FACTOR VEHICLE 5,c_COLLISION_ID,c_VEHICLE TYPE CODE 1,c_VEHICLE TYPE CODE 2,c_VEHICLE TYPE CODE 3,c_VEHICLE TYPE CODE 4,c_VEHICLE TYPE CODE 5,v_UNIQUE_ID,v_STATE_REGISTRATION,v_VEHICLE_TYPE,v_VEHICLE_MAKE,v_VEHICLE_MODEL,v_VEHICLE_YEAR,v_TRAVEL_DIRECTION,v_VEHICLE_OCCUPANTS,v_DRIVER_SEX,v_DRIVER_LICENSE_STATUS,v_DRIVER_LICENSE_JURISDICTION,v_PRE_CRASH,v_POINT_OF_IMPACT,v_VEHICLE_DAMAGE,v_VEHICLE_DAMAGE_1,v_VEHICLE_DAMAGE_2,v_VEHICLE_DAMAGE_3,v_PUBLIC_PROPERTY_DAMAGE,v_PUBLIC_PROPERTY_DAMAGE_TYPE,v_CONTRIBUTING_FACTOR_1,v_CONTRIBUTING_FACTOR_2,p_UNIQUE_ID,p_PERSON_ID,p_PERSON_TYPE,p_PERSON_INJURY,p_PERSON_AGE,p_EJECTION,p_EMOTIONAL_STATUS,p_BODILY_INJURY,p_POSITION_IN_VEHICLE,p_SAFETY_EQUIPMENT,p_PED_LOCATION,p_PED_ACTION,p_COMPLAINT,p_PED_ROLE,p_CONTRIBUTING_FACTOR_1,p_CONTRIBUTING_FACTOR_2,p_PERSON_SEX,HAS_VEHICLE,HAS_PERSON
0,2023-11-01,01:29:00,BROOKLYN,11230,40.621790,-73.970024,"(40.62179, -73.970024)",1.0,0.0,0,0,0,0,1,0,Unspecified,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN,20542014.0,NY,Sedan,NISS -CAR/SUV,NaN,2022.0,South,1,F,Licensed,NY,Going Straight Ahead,Center Front End,Center Front End,Undercarriage,Undercarriage,Left Front Bumper,N,NaN,Unspecified,Unspecified,12786837.0,7c43fd06-96f8-4564-89dc-48543318b6ae,Occupant,Unspecified,63,Not Ejected,Does Not Apply,Does Not Apply,Driver,Lap Belt,NaN,NaN,Does Not Apply,Driver,NaN,NaN,F,1,1
1,2023-11-01,01:29:00,BROOKLYN,11230,40.621790,-73.970024,"(40.62179, -73.970024)",1.0,0.0,0,0,0,0,1,0,Unspecified,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN,20542013.0,NaN,Sedan,NaN,NaN,NaN,South,1,NaN,NaN,NaN,Going Straight Ahead,NaN,NaN,NaN,NaN,NaN,N,NaN,Unspecified,Driver Inattention/Distraction,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,2023-11-01,01:29:00,BROOKLYN,11230,40.621790,-73.970024,"(40.62179, -73.970024)",1.0,0.0,0,0,0,0,1,0,Unspecified,Unspecified,Unspecified,NaN,NaN,4675373,Moped,Sedan,Sedan,NaN,NaN,20542012.0,NaN,Moped,NaN,NaN,NaN,South,1,M,Permit,NY,Going Straight Ahead,Center Back End,Center Back End,Center Back End,Center Back End,Center Back End,N,NaN,Unspecified,Unspecified,12786836.0,0f7c79da-748a-4bae-a24a-6dff0e0d2d19,Occupant,Injured,26,Ejected,Conscious,Knee-Lower Leg Foot,Driver,NaN,NaN,NaN,Complaint of Pain or Nausea,Driver,NaN,NaN,M,1,1
3,2021-09-11,09:35:00,BROOKLYN,11208,40.667202,-73.866500,"(40.667202, -73.8665)",0.0,0.0,0,0,0,0,0,0,Unspecified,NaN,NaN,NaN,NaN,4456314,Sedan,NaN,NaN,NaN,NaN,20060293.0,NC,Sedan,DODG -CAR/SUV,NaN,2018.0,South,0,F,Licensed,NC,Parked,Left Rear Quarter Panel,Left Rear Quarter Panel,Left Side Doors,Left Front Bumper,NaN,N,NaN,Unspecified,Unspecified,11945216.0,3cb21800-426f-47c8-a79e-fb65f2d2115e,Occupant,Unspecified,28,Not Ejected,Does Not Apply,Does Not Apply,Unknown,NaN,NaN,NaN,Does Not Apply,Driver,NaN,NaN,F,1,1
4,2021-09-11,09:35:00,BROOKLYN,11208,40.667202,-73.866500,"(40.667202, -73.8665)",0.0,0.0,0,0,0,0,0,0,Unspecified,NaN,NaN,NaN,NaN,4456314,Sedan,NaN,NaN,NaN,NaN,20060293.0,NC,Sedan,DODG -CAR/SUV,NaN,2018.0,South,0,F,Licensed,NC,Parked,Left Rear Quarter Panel,Left Rear Quarter Panel,Left Side Doors,Left Front Bumper,NaN,N,NaN,Unspecified,Unspecified,11945217.0,d7bbe88a-d44d-4155-8076-923b24b371be,Occupant,Unspecified,28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Registrant,NaN,NaN,F,1,1


In [86]:
# save as a new csv file
full_df.to_csv('datasets/crash_vehicle_person_merged_data.csv', index=False)

In [87]:
# save to parquet
full_df.to_parquet('datasets/crash_vehicle_person_merged_data.parquet', engine= 'pyarrow', index=False)

In [88]:
# save the list of columns to a text file
with open('datasets/merged_data_columns_list.txt', 'w') as f:
    for col in full_df.columns:
        f.write(f"{col}\n")

In [89]:
full_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6836842 entries, 0 to 6836841
Data columns (total 66 columns):
 #   Column                           Dtype  
---  ------                           -----  
 0   c_CRASH DATE                     object 
 1   c_CRASH TIME                     object 
 2   c_BOROUGH                        str    
 3   c_ZIP CODE                       int64  
 4   c_LATITUDE                       float64
 5   c_LONGITUDE                      float64
 6   c_LOCATION                       str    
 7   c_NUMBER OF PERSONS INJURED      float64
 8   c_NUMBER OF PERSONS KILLED       float64
 9   c_NUMBER OF PEDESTRIANS INJURED  int64  
 10  c_NUMBER OF PEDESTRIANS KILLED   int64  
 11  c_NUMBER OF CYCLIST INJURED      int64  
 12  c_NUMBER OF CYCLIST KILLED       int64  
 13  c_NUMBER OF MOTORIST INJURED     int64  
 14  c_NUMBER OF MOTORIST KILLED      int64  
 15  c_CONTRIBUTING FACTOR VEHICLE 1  str    
 16  c_CONTRIBUTING FACTOR VEHICLE 2  str    
 17  c_CONTRIBUTING FACT